In [1]:
import os
import glob
from dotenv import load_dotenv
from pathlib import Path
from openai import OpenAI
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


## Get all filepaths

In [2]:
from pathlib import Path

filepath_dct = {}  # category -> list of filepath
category_ls = []

base_dir = "./fudan_news_dataset/fudan_news/Fudan/train/"
for filename in Path(base_dir).iterdir():
    if not filename.is_dir(): continue
        
    topic = filename.name.split('-')[-1]
    category_ls.append(topic)
    filepath_dct[topic] = []
    for fname in Path(filename, 'utf8').iterdir():
        if fname.is_dir(): continue
            
        filepath_dct[topic].append(fname)


In [63]:
print("List of domains from the text dataset: \n{}".format('\n'.join(category_ls)))


List of domains from the text dataset: 
Art
Literature
Economy
Mine
Sports
History
Computer
Transport
Energy
Politics
Space
Communication
Agriculture
Education
Military
Electronics
Medical
Philosophy
Law
Enviornment


## Get top vocabularies for each domain

In [4]:
import jieba
import jieba.posseg as pseg
from collections import defaultdict
from tqdm import tqdm


In [5]:
with open("cn_stopwords.txt", encoding="utf-8") as f:
    stopwords = set([line.strip() for line in f if line.strip()])

In [6]:
def is_all_chinese(word):
    return all('\u4e00' <= char <= '\u9fff' for char in word)

In [8]:
def calculate_words(fp, words_dct):
    """
    - read text file, remove entity, stopwords, and non-chinese words
    - get vocabs count and update words_dct
    """
    ner_tags = {'ns', 'nr', 'nt', 'eng'}
    with open(fp, 'r') as f:
        text = f.read()
        # words = list(jieba.cut(text, cut_all=False))
        words = pseg.lcut(text)
        
        for word, flag in words:
            if flag not in ner_tags and is_all_chinese(word) and word not in stopwords:
                words_dct[word] += 1


In [9]:
# process all files and get words stats

topic_words_dct = {}

for topic in filepath_dct.keys():
    words_dct = defaultdict(int)
    topic_words_dct[topic] = words_dct
    print(f"process topic: {topic}")
    
    for fp in tqdm(filepath_dct[topic]):
        calculate_words(fp, words_dct)


process topic: Art


  0%|                                                                                     | 0/740 [00:00<?, ?it/s]Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/15/m36n7q250r32w2hg98gyqckr0000gn/T/jieba.cache
Loading model cost 0.445 seconds.
Prefix dict has been built successfully.
100%|███████████████████████████████████████████████████████████████████████████| 740/740 [02:01<00:00,  6.10it/s]


process topic: Literature


100%|█████████████████████████████████████████████████████████████████████████████| 33/33 [00:00<00:00, 38.50it/s]


process topic: Economy


100%|█████████████████████████████████████████████████████████████████████████| 1600/1600 [02:22<00:00, 11.27it/s]


... (30 outputs left out)

process topic: Law


100%|████████████████████████████████████████████████████████████████████████████| 51/51 [00:00<00:00, 148.53it/s]


process topic: Enviornment


100%|█████████████████████████████████████████████████████████████████████████| 1217/1217 [01:17<00:00, 15.61it/s]


In [10]:
def get_topic_with_word(topic_words_dct, word):
    count = 0
    for words_dct in topic_words_dct.values():
        if word in words_dct:
            count += 1
    return count
    

In [11]:
# get tf-idf
import math

topic_tf_idf_dct = {}
n_topic = len(topic_words_dct.keys())
for topic, words_dct in topic_words_dct.items():
    topic_tf_idf_dct[topic] = {}
    total_words = sum(list(words_dct.values()))

    for word in words_dct.keys():
        tf = words_dct[word]/total_words
        
        topic_with_that_word = get_topic_with_word(topic_words_dct, word)
        
        idf = math.log(n_topic/topic_with_that_word)

        topic_tf_idf_dct[topic][word] = tf*idf


In [12]:
# get top 100 words per topic by tf-idf
top_words_dct = {}
for topic, tf_idf_dct in topic_tf_idf_dct.items():
    top_words_dct[topic] = sorted(list(tf_idf_dct.items()), key=lambda x: x[1], reverse=True)[:100]


In [49]:
print("Top 20 vocabularies for Computer topic")
print('\n'.join([x[0] for x in top_words_dct['Computer'][:20]]))

Top 20 vocabularies for Computer topic
算法
模型
函数
定义
服务器
数据
控制器
矩阵
图
本文
结点
数据库
参数
描述
引理
文档
给出
神经网络
对象
输出


In [64]:
# save top vocabularies to file
import json

with open("top_vocabs.json", "w") as f:
    json.dump(top_words_dct, f, indent=4)

In [ ]:
# load if needed
with open("top_vocabs.json", "r", encoding="utf-8") as f:
    top_words_dct = json.load(f)
    

## Use RAG, process documents and save to vector db

In [32]:
base_dir = "./fudan_news_dataset/fudan_news/Fudan/train/*"
path_dct = {path.split('/')[-1].split('-')[-1]: os.path.join(path, 'utf8') for path in glob.glob(base_dir)}

In [54]:
def doc_chunking_text(fp):
    loader = TextLoader(fp)
    documents = loader.load()
    
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks = text_splitter.split_documents(documents)

    return chunks

In [33]:
def doc_chunking(folder):
    documents = []
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="*.txt", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

    print(f"Loaded {len(documents)} documents")

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks = text_splitter.split_documents(documents)

    return chunks

In [21]:
def put_chunk_in_db(db_name, chunks, embeddings):
    if os.path.exists(db_name):
        Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)


In [34]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# embeddings = OpenAIEmbeddings(model="text-embedding-3-large")


/Users/delphinehelenanatasha/opt/anaconda3/envs/basic/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [35]:
for topic, fp_ls in filepath_dct.items():
    print(f"Process topic {topic}")
    db_name = "db_" + topic
    folder = path_dct[topic]
    chunks = doc_chunking(folder)
    put_chunk_in_db(db_name, chunks, embeddings)
    
    # for fp in tqdm(fp_ls):
    #     chunks = doc_chunking(fp)
    #     put_chunk_in_db(db_name, chunks, embeddings)

Process topic Art
Loaded 740 documents
Process topic Literature
Loaded 33 documents
Process topic Economy
Loaded 1600 documents
Process topic Mine
Loaded 33 documents
Process topic Sports
Loaded 1253 documents
Process topic History
Loaded 465 documents
Process topic Computer
Loaded 1357 documents
Process topic Transport
Loaded 57 documents
Process topic Energy
Loaded 32 documents
Process topic Politics
Loaded 1024 documents
Process topic Space
Loaded 640 documents
Process topic Communication
Loaded 25 documents
Process topic Agriculture
Loaded 1021 documents
Process topic Education
Loaded 59 documents
Process topic Military
Loaded 74 documents
Process topic Electronics
Loaded 27 documents
Process topic Medical
Loaded 51 documents
Process topic Philosophy
Loaded 44 documents
Process topic Law
Loaded 51 documents
Process topic Enviornment
Loaded 1217 documents


## Init LLM

In [16]:
def init_env():
    load_dotenv(override=True)
    openai_api_key = os.getenv('OPENAI_API_KEY')
    if openai_api_key:
        print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
    else:
        print("OpenAI API Key not set")

    MODEL = "gpt-6-luna"
    openai = OpenAI()
    return openai, MODEL

In [17]:
openai, MODEL = init_env()

OpenAI API Key exists and begins sk-proj-


## Random sampling from top words and ask LLM to explain

In [14]:
import random


In [15]:
topic = 'Computer'
K = 5

words = [x[0] for x in top_words_dct[topic]]
random_words = random.sample(words, K)
print(random_words)


['查询', '分布', '定义', '方法', '输入']


In [52]:
def explain(openai, MODEL, words, topic):
    SYSTEM_PREFIX = """
    You are a helpful Chinese language tutor. You can explain the meaning and example of a given list of words.
    """
    system_message = SYSTEM_PREFIX
    words_str = ', '.join(words)
    user_message = f"Please explain the meaning of the following Chinese words in English and give short sentence as example of the usage in {topic} domain: {words_str}"
    messages = [{"role": "system", "content": system_message}] + [{"role": "user", "content": user_message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [57]:
print(explain(openai, MODEL, random_words, topic))


| Word | Meaning | Computer-domain example |
|---|---|---|
| **查询** (cháxún) | to query; to look up or retrieve information | **你可以查询数据库中的用户信息。** — You can query user information in the database. |
| **分布** (fēnbù) | distribution; to be distributed | **这些文件分布在多个服务器上。** — These files are distributed across multiple servers. |
| **定义** (dìngyì) | to define; a definition | **请在代码中定义一个变量。** — Please define a variable in the code. |
| **方法** (fāngfǎ) | method; approach; in programming, a method | **这个类有一个读取文件的方法。** — This class has a method for reading files. |
| **输入** (shūrù) | input; to enter or input data | **请在搜索框中输入关键词。** — Please enter a keyword in the search box. |


## Ask LLM to create conversation without RAG

In [18]:
def chat(openai, MODEL, words):
    SYSTEM_PREFIX = """
    You are a helpful Chinese language tutor. You can provide a good example of conversation for a given list of words.
    """
    system_message = SYSTEM_PREFIX
    user_message = "Please generate a short conversation in simplified Chinese and the English translation using the following words: {}".format(', '.join(words))
    messages = [{"role": "system", "content": system_message}] + [{"role": "user", "content": user_message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content
    

In [19]:
text1 = chat(openai, MODEL, random_words)
print(text1)

**中文**

小林：你知道怎么查询这些城市的人口分布吗？  
小王：先输入城市名称，再选择查询方法。  
小林：页面上“分布”的定义是什么？  
小王：就是人口在不同地区的分配情况。

**English**

Xiaolin: Do you know how to look up the population distribution of these cities?  
Xiaowang: First enter the city name, then choose a search method.  
Xiaolin: What does “distribution” mean on this page?  
Xiaowang: It means how the population is spread across different areas.


## Ask LLM to create conversation with content from RAG

In [36]:
def retrieve_content(db_name, words, embeddings):
    vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)
    retriever = vectorstore.as_retriever()
    query = ", ".join(words)
    docs = retriever.invoke(query)
    context = "\n\n".join(doc.page_content for doc in docs)
    return context

In [38]:
context = retrieve_content("db_Computer", random_words, embeddings)

In [39]:
context

'图2  与数据库相连的方法对数据库的操作\n\n算法. （辩论语义的迭代算法）\n\n图 3\u3000加工实例\n\n图 4\u3000一般方法处理结果\n\n[2] 赵丽萍， 何钺.基于功能特征产品模型定义方法研究.计算机辅助工程，1997， 6(1).\n[3] 赵丽萍等.装配偶件的功能特征建模方法研究.机械工业自动化，1996，18(1).\n收稿日期\u30001995-07-21'

In [45]:
def chat_with_rag(openai, MODEL, words, context):
    SYSTEM_PREFIX = """
    You are a helpful Chinese language tutor. You can provide a good example of conversation for a given list of words.
    You are provided with additional texts that you must use to generate the example of conversation.

    Relevant context:
    """
    system_message = SYSTEM_PREFIX + context
    # print(system_message)
    user_message = "Please generate a short conversation in simplified Chinese and the English translation using the following words: {}. Please refers to the context above for the conversation style and topic.".format(', '.join(words))
    messages = [{"role": "system", "content": system_message}] + [{"role": "user", "content": user_message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [46]:
text2 = chat_with_rag(openai, MODEL, random_words, context)
print(text2)

**中文**

甲：我们先输入加工实例，再用与数据库相连的方法查询相关数据。  
乙：好的。这个模型的定义是什么？  
甲：它根据产品的功能特征进行定义，查询结果会显示特征的分布。

**English**

A: Let’s first input a processing example, then use the database-linked method to query the relevant data.  
B: All right. What is the definition of this model?  
A: It is defined according to the product’s functional features, and the query results show how those features are distributed.
